In [100]:
import re
from re import findall

str=" afd [asd] [12 ] [a34] [ -43 ]tt [+12]xxx"
print(re.findall(r'\[\s*([+-]?\d+)\s*\]', str))

['12', '-43', '+12']


Exercise 2.2 (file listing)

The file src/listing.txt contains a list of files with one line per file. Each line contains seven fields: access rights, number of references, owner's name, name of owning group, file size, date, filename. These fields are separated with one or more spaces. Note that there may be spaces also within these seven fields.

Write function file_listing that loads the file src/listing.txt. It should return a list of tuples (size, month, day, hour, minute, filename). Use regular expressions to do this (either match, search, findall, or finditer method).

An example: for line

-rw-r--r-- 1 jttoivon hyad-all   25399 Nov  2 21:25 exception_hierarchy.pdf
the function should create the tuple (25399, "Nov", 2, 21, 25, "exception_hierarchy.pdf").

In [ ]:
#!/usr/bin/env python3

import re


def file_listing(filename="src/listing.txt"):
    """Parse a `ls -l` style listing into (size, month, day, hour, minute, filename) tuples.

    Example line "-rw-r--r-- 1 jttoivon hyad-all 25399 Nov  2 21:25 x.pdf"
    produces (25399, 'Nov', 2, 21, 25, 'x.pdf').
    """

    pattern = re.compile(
        r'^\S+\s+\d+\s+\S+\s+\S+\s+'   # rights, refs, owner, group (skipped)
        r'(\d+)\s+'                     # size
        r'(\w+)\s+(\d+)\s+'             # month, day
        r'(\d+):(\d+)\s+'               # hour:minute
        r'(.+)$'                        # filename (rest of the line)
    )

    result = []
    with open(filename) as f:
        for line in f:
            m = pattern.match(line)
            if m:
                size, month, day, hour, minute, fname = m.groups()
                result.append((int(size), month, int(day), int(hour), int(minute), fname))
    return result


def main():
    print(file_listing())

if __name__ == "__main__":
    main()


## How I Built the Pattern — Step by Step

Let me walk through the actual thought process, the way you'd approach it yourself.

---

### Step 1 — Look at the Raw Data First

```
-rw-r--r-- 1 jttoivon hyad-all   25399 Nov  2 21:25 exception_hierarchy.pdf
-rwxr-xr-x 1 jttoivon hyad-all    2356 Dec 11 11:50 add_colab_link.py
```

I count the **fields** the exercise promised (7 of them) and label each one:

```
-rw-r--r--   1   jttoivon   hyad-all   25399   Nov  2   21:25   exception_hierarchy.pdf
   rights    refs  owner      group     size   month day  time      filename
```

**Key observation:** the exercise only wants **6 of these** back: size, month, day, hour, minute, filename. Rights, refs, owner, group are noise I need to **step over**, not capture.

---

### Step 2 — Classify Each Field by "What Kind of Text Is This?"

For every field, I ask: *is this a number, a word, or just any-non-space stuff?* — because that decides which regex building block fits.

| Field | What it looks like | Building block |
|---|---|---|
| rights | `-rw-r--r--` | any non-space chunk | `\S+` |
| refs | `1` | a number, but I don't need it | `\S+` (don't bother being precise — I'm discarding it) |
| owner | `jttoivon` | any non-space chunk | `\S+` |
| group | `hyad-all` | any non-space chunk | `\S+` |
| **size** | `25399` | digits — **I need this** | `(\d+)` |
| **month** | `Nov` | letters — **I need this** | `(\w+)` |
| **day** | `2` | digits — **I need this** | `(\d+)` |
| **hour** | `21` | digits — **I need this** | `(\d+)` |
| **minute** | `25` | digits — **I need this** | `(\d+)` |
| **filename** | `exception_hierarchy.pdf` | rest of the line — **I need this** | `(.+)$` |

**Decision rule I applied:** fields I *keep* get wrapped in `(...)` (capture); fields I *discard* stay bare `\S+` (match but don't remember).

---

### Step 3 — Figure Out the Separators

I look at the raw text again, very literally, character by character:

```
25399[space][space]Nov[space][space]2[space]21:25[space]exception...
```

Two things jump out:
1. There's **inconsistent spacing** — sometimes 1 space, sometimes 2 or 3 (right-aligned columns)
2. Hour and minute are joined by a **literal colon**, not a space: `21:25`

So:
- Anywhere I see "some amount of space" → `\s+` (one-or-more), never a single literal `' '`
- Between hour and minute specifically → a literal `:`

---

### Step 4 — Assemble Left to Right, Matching the Line's Actual Order

I just walk the line from left to right and write down, piece by piece, what I see:

```
start of line
 → rights (skip)         \S+
 → space(s)               \s+
 → refs (skip)            \d+          (could be \S+ too, either works)
 → space(s)               \s+
 → owner (skip)           \S+
 → space(s)               \s+
 → group (skip)           \S+
 → space(s)               \s+
 → size (KEEP)             (\d+)
 → space(s)               \s+
 → month (KEEP)            (\w+)
 → space(s)               \s+
 → day (KEEP)              (\d+)
 → space(s)               \s+
 → hour (KEEP)             (\d+)
 → literal colon           :
 → minute (KEEP)           (\d+)
 → space(s)               \s+
 → filename (KEEP)         (.+)
 → end of line             $
```

Concatenating all of that, left to right, in order, IS the final pattern:

```python
r'^\S+\s+\d+\s+\S+\s+\S+\s+(\d+)\s+(\w+)\s+(\d+)\s+(\d+):(\d+)\s+(.+)$'
```

Nothing more clever than that — I described the line piece by piece, exactly as I saw it, and let the regex mirror the description.

---

### Step 5 — Anchor It (the `^` and `$`)

I added `^` at the start and `$` at the end **on purpose** — recall from your `\A`/`\Z`/`^`/`$` lesson: this pins the pattern to the **whole line**. Without `^`, the regex could theoretically start matching from some accidental position mid-line; without `$`, the filename capture `(.+)` might stop early instead of eating the rest of the line. Anchoring both ends makes the match behave predictably, one line = one match attempt.

---

### Step 6 — Choose `.+` (Not `\S+`) for the Filename — Deliberately

This was the one place I paused and thought carefully, because two options seemed plausible:

```
\S+   → "filename has no spaces"        — WRONG per the exercise's own warning!
(.+)  → "filename is everything left"   — matches the "spaces within fields" warning
```

The exercise explicitly said: *"there may be spaces also within these seven fields."* That sentence is a direct hint — it's telling me not to assume filenames are space-free. So I used `(.+)$` — greedy, but safe here because it's the **last** field; there's nothing after it to accidentally overshoot into (unlike the earlier `.* , then` trap, where greed ate too far because there WAS more text after the intended stopping point).

---

### Step 7 — Test It Mentally Against a Real Line

Before trusting it, I traced it by hand against your sample:

```
-rw-r--r-- 1 jttoivon hyad-all   25399 Nov  2 21:25 exception_hierarchy.pdf
```

```
\S+     → "-rw-r--r--"        ✓
\s+     → " "                 ✓
\d+     → "1"                 ✓
\s+     → " "                 ✓
\S+     → "jttoivon"          ✓
\s+     → " "                 ✓
\S+     → "hyad-all"          ✓
\s+     → "   " (3 spaces!)   ✓ (\s+ absorbs any amount)
(\d+)   → "25399"             ✓ captured
\s+     → " "                 ✓
(\w+)   → "Nov"                ✓ captured
\s+     → "  " (2 spaces!)     ✓ absorbed
(\d+)   → "2"                  ✓ captured
\s+     → " "                  ✓
(\d+)   → "21"                 ✓ captured
:       → ":"                  ✓ literal
(\d+)   → "25"                 ✓ captured
\s+     → " "                  ✓
(.+)$   → "exception_hierarchy.pdf"  ✓ captured, to end of line
```

Everything lined up — six groups, matching exactly the six values the exercise wanted.

---

### The General Method, Distilled

> 1. **Print a sample line and label every field** by hand.
> 2. **Decide keep vs. discard** — captured fields get `(...)`, discarded fields stay bare.
> 3. **Classify each field's character type** — digits → `\d+`, words → `\w+`, "don't care, just skip" → `\S+`, "rest of the line" → `.+`.
> 4. **Look at the actual separators** in the raw text — spacing amount, literal punctuation like `:` — and match them literally.
> 5. **Assemble left to right**, mirroring the line's real structure.
> 6. **Anchor** with `^`/`$` if you want the pattern to own the whole line.
> 7. **Trace it by hand** against a real example before trusting it.
>
> Building a regex isn't about memorizing a formula — it's a translation exercise: describe what you see in the text, one chunk at a time, then let the description become the pattern. 🎯

What is the different between the way you wrote this

## Two Different Ways to WRITE the Same Pattern

Good catch — **they are functionally identical**, byte for byte the same regex. The difference is purely about **readability of the source code**, not about what the pattern does.

---

### The Trick — Adjacent String Literals Auto-Concatenate

This is a Python feature you haven't explicitly met yet: when you write **several string literals next to each other** with nothing but whitespace/newlines between them, Python **silently joins them into one string**:

```python
s = "Hello" " " "World"
print(s)          # → "Hello World"
print(s == "Hello World")   # → True
```

No `+`, no `.join()` — just placing them side by side is enough. This works for raw strings too (`r"..."`), which is exactly what regex patterns use.

---

### Proving Both Versions Produce the SAME String

```python
version_a = (
    r'^\S+\s+\d+\s+\S+\s+\S+\s+'
    r'(\d+)\s+'
    r'(\w+)\s+(\d+)\s+'
    r'(\d+):(\d+)\s+'
    r'(.+)$'
)

version_b = r'^\S+\s+\d+\s+\S+\s+\S+\s+(\d+)\s+(\w+)\s+(\d+)\s+(\d+):(\d+)\s+(.+)$'

print(version_a == version_b)   # → True !
```

Python glues the four separate `r'...'` chunks in `version_a` into **one single string**, identical to `version_b`. `re.compile()` never even sees the difference — it receives the exact same text either way.

---

### Why Split It Into Pieces at All? — Readability, With Comments Attached

The multi-line version's real purpose is letting you attach a **comment to each logical piece**:

```python
pattern = re.compile(
    r'^\S+\s+\d+\s+\S+\s+\S+\s+'   # rights, refs, owner, group (skipped)
    r'(\d+)\s+'                     # size
    r'(\w+)\s+(\d+)\s+'             # month, day
    r'(\d+):(\d+)\s+'               # hour:minute
    r'(.+)$'                        # filename (rest of the line)
)
```

Regular Python code **can't put a `#` comment in the middle of a single string literal** — `#` inside quotes is just a literal character, not a comment marker:

```python
r'abc # this is NOT a comment, it is part of the string!'
```

So the **only** way to attach an inline explanation to *part* of a regex is to **break the string into separate literal pieces** at the exact boundaries you want to comment — then Python's auto-concatenation glues them back into one pattern behind the scenes, while each piece keeps its own trailing comment.

---

### Side-by-Side Comparison

| | One-liner | Split with comments |
|---|---|---|
| Actual regex string | identical | identical |
| Runtime behavior | identical | identical |
| Readability for humans | you must decode the whole blob at once | each chunk is pre-labeled |
| Best for | short, simple patterns | long patterns with many distinct parts (like yours!) |

Given that your pattern has **7 distinct fields** (4 skipped + size + month/day + hour/minute + filename), the split version is genuinely easier to review later — you or a teammate can scan the comments without mentally re-parsing 60+ characters of `\s+` and `\d+`.

---

### One More Option Worth Knowing — `re.VERBOSE`

There's actually a **third, even cleaner way** for long patterns: the `re.VERBOSE` flag lets you write the pattern with **real whitespace and comments inside a single raw string**, and tells the engine to ignore that formatting:

```python
pattern = re.compile(r"""
    ^\S+ \s+ \d+ \s+ \S+ \s+ \S+ \s+   # rights, refs, owner, group (skipped)
    (\d+) \s+                          # size
    (\w+) \s+ (\d+) \s+                # month, day
    (\d+):(\d+) \s+                    # hour:minute
    (.+) $                             # filename
""", re.VERBOSE)
```

This achieves the same readability goal as your split version, but in **one string** instead of four concatenated ones — some people find it even clearer since the whole pattern reads top-to-bottom like a mini-program. (Caveat: with `VERBOSE`, literal spaces in the pattern must be escaped as `\ ` or placed inside `[ ]`, since plain whitespace is now ignored — not an issue here since we already use `\s+` everywhere instead of literal spaces.)

---

### The One-Sentence Summary

> Both versions compile to **the exact same pattern** — Python automatically concatenates adjacent string literals, so splitting `r'...'` across multiple lines is purely a way to attach a `#` comment to each chunk (which you can't do inside a single string). Choose the split style for long, multi-part patterns where labeling each piece helps future-you; use the one-liner for short, self-explanatory patterns. 🎯